# 12. More outlets in training (added 8/11)

**Question.** The limit seems to be the number of independent outlets (about 180 per fold). If the non-political
NewsGuard outlets we scraped are added to the TRAINING (the test stays the 226 political outlets, same folds and same
draws), do the models predict the political outlets better?

Everything is in this notebook: the extra dataset (same cleaning and anonymization rules as steps 2-3, copies of political
articles removed, ownership groups kept isolated, at most 30 articles per extra outlet), the frozen vectors of the extra
articles, the models (TF-IDF, Qwen3-8B and Mistral-24B with ridge, outlet-weighted ridge and the small neural network),
the comparison with the same models without extra outlets (steps 7 and 10). Rules and reasons in `more_outlets.py`.
The GPU part waits for the step 11 queue to finish (`WAIT_FOR`). Every piece is saved, so a re-run resumes.

In [1]:
N_JOBS = 48
RUN_BERT = False                      # ModernBERT on meta-articles with the extra outlets (as step 11), only on request
WAIT_FOR = "Results/meta_done"        # the GPU part starts after this file exists (step 11 queue finished)

In [2]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())
import time
import traceback
from pathlib import Path

import matplotlib.pyplot as plt

import anonymization as AN
import boost as BO
import boost_eval as BE
import more_outlets as MX

MX.OUT.mkdir(parents=True, exist_ok=True)
def safely(label, f, *a):
    """A failing piece is logged and the notebook goes on (a re-run retries only what is missing)."""
    try:
        f(*a)
    except Exception as e:
        C.log(f"ERROR {label}: {type(e).__name__}: {e}", MX.LOG)
        with open(MX.OUT / "errors.txt", "a") as fh:
            fh.write(f"==== {label}\n{traceback.format_exc()}\n")

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


## 12.1 The extra dataset

In [3]:
steps = []
def step(name, df):
    steps.append({"step": name, "articles": len(df), "outlets": df.domain.nunique()})
    print(f"{name}: {len(df):,} articles, {df.domain.nunique():,} outlets")

final = MX.OUT / "articles_extra_anonymous.csv.gz"
if not final.exists():
    ng, cand = MX.candidates()
    print(f"candidate outlets (NewsGuard score, not political-only): {len(cand):,}")
    raw = MX.extract(cand, N_JOBS); step("scraped articles of the candidates", raw)
    art = MX.filters(raw, N_JOBS, step)
    political = pd.read_csv(C.OUT / "articles.csv.gz")
    art = MX.deduplicate(art, political, N_JOBS, step)
    art = MX.groups_and_cap(art, sorted(political.domain.unique()), ng, step)
    anon, counts, found = AN.anonymize(art)
    left = AN.leftovers(anon, found)
    RULES = ["signature", "email", "url", "account", "author", "name"]
    print(f"anonymization: articles touched {(counts[RULES].sum(axis=1) > 0).mean():.1%}; "
          f"distinctive names left as a word: {left[~left.common_word].left_as_word.sum()}")
    anon[["article_id", "domain", "title", "date", "text", "score", "label"]].to_csv(final, index=False)
    pd.DataFrame(steps).to_csv(MX.OUT / "steps.csv", index=False)
pd.read_csv(MX.OUT / "steps.csv").set_index("step")

,articles,outlets
step,,
scraped articles of the candidates,206910,1700
with extracted text,204760,1696
English,132796,1371
"100-1,500 words",110715,1340
published from 2025,89359,1241
no exact copies (of political or extra articles),78102,1208
no near copies (Jaccard >= 0.5; 281 were copies of political articles),70348,1100
outlets with >= 10 articles,69043,772
not in the ownership group of a political outlet,67252,754


In [4]:
extra = pd.read_csv(final)
U, O, rows, outlets, extra_outlets = MX.union()
eo = O.loc[extra_outlets]
print(f"extra: {len(extra_outlets)} outlets, {len(extra):,} articles; political: {len(outlets)} outlets, {(U.article_id < MX.EXTRA_ID).sum():,} articles")
pd.DataFrame({"political outlets": outlets.label.value_counts(), "extra outlets": eo.label.value_counts()}).reindex(range(5)).rename(index=dict(enumerate(C.LEVELS))).T

/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


extra: 232 outlets, 6,534 articles; political: 226 outlets, 25,524 articles


label,l1,l2,l3,l4,l5
political outlets,35,43,30,79,39
extra outlets,57,35,25,75,40


## 12.2 TF-IDF (CPU)

In [5]:
for f in range(1, 6):
    safely(f"TF-IDF fold {f}", MX.fit_tfidf, f, U, rows, outlets, extra_outlets, N_JOBS)

## 12.3 Frozen vectors and models (GPU, after the step 11 queue)

In [6]:
while not Path(WAIT_FOR).exists():
    time.sleep(300)
for name in ["Qwen3-8B", "Mistral-24B"]:
    holder = {}
    safely(f"embed {name}", lambda: holder.update(X=MX.embed(name, U)))
    if "X" not in holder:
        continue
    X = holder["X"]
    for method in ["ridge", "ridge_outlet", "mlp"]:
        for f in range(1, 6):
            safely(f"{name} · {method} fold {f}", MX.fit_embedding, name, method, f, X, U, O, rows, outlets, extra_outlets)
    del X

In [7]:
if RUN_BERT:
    import meta as ME
    M, _ = ME.load_meta()
    art_pol, _, _ = BO.load()
    for f in range(1, 6):
        safely(f"ModernBERT fold {f}", ME.fit_bert, f, M, art_pol, outlets, extra, extra.groupby("domain").score.first(), MX.PRED)

## 12.4 With and without the extra outlets

In [8]:
done = [m for m in MX.MODELS if len(list((MX.PRED / m).glob("fold*.csv.gz"))) == 5]
print("complete:", done)
err = MX.OUT / "errors.txt"
print(open(err).read()[-3000:] if err.exists() else "no errors")
idx = np.vstack([np.arange(len(outlets))[None, :], np.random.default_rng(7).integers(0, len(outlets), (BE.B, len(outlets)))])
rows_out = []
for m in done:
    for k in ["1", "5", "all"]:
        with_ = BE._one(MX.matrix(m, outlets, k, True), m, k, outlets, idx)[2]
        without = BE._one(MX.matrix(m, outlets, k, False), m, k, outlets, idx)[2]
        for (score, mes), v in with_.items():
            d = v - without[(score, mes)]; lo, hi = np.percentile(d[1:], [2.5, 97.5])
            rows_out.append({"model": m, "k": k, "score": score, "measure": mes, "without": without[(score, mes)][0],
                             "with extra": v[0], "difference": d[0], "low": lo, "high": hi,
                             "better": bool(hi < 0) if mes in BE.LOWER else bool(lo > 0),
                             "worse": bool(lo > 0) if mes in BE.LOWER else bool(hi < 0)})
T = pd.DataFrame(rows_out); T.to_csv(MX.OUT / "results.csv", index=False)

complete: ['TF-IDF', 'Qwen3-8B · ridge', 'Qwen3-8B · ridge_outlet', 'Qwen3-8B · mlp', 'Mistral-24B · ridge', 'Mistral-24B · ridge_outlet', 'Mistral-24B · mlp']
no errors


In [9]:
def table(k="all", score="stretched"):
    """without → with extra outlets; ✓ significantly better, ✗ significantly worse (paired bootstrap)."""
    d = T[(T.k == k) & (T.score == score)].copy()
    f = lambda v, m: f"{v:.3f}" if m in ("Spearman", "Kendall") else f"{v:.1f}"
    d["text"] = [f"{f(a, m)} → {f(b, m)}" + (" ✓" if g else " ✗" if w else "") for a, b, m, g, w in
                 zip(d.without, d["with extra"], d.measure, d.better, d.worse)]
    return d.pivot_table(index="model", columns="measure", values="text", aggfunc="first")[BE.MEASURES].reindex(done)


table("all", "stretched")

measure,Spearman,Kendall,MAE,within 20
model,,,,
TF-IDF,0.696 → 0.710,0.518 → 0.529,16.3 → 16.5,70.8 → 68.6
Qwen3-8B · ridge,0.726 → 0.739,0.544 → 0.563,15.3 → 14.6,75.7 → 74.3
Qwen3-8B · ridge_outlet,0.726 → 0.723,0.543 → 0.543,15.6 → 15.4,73.0 → 71.7
Qwen3-8B · mlp,0.718 → 0.719,0.539 → 0.544,15.7 → 15.1,76.1 → 73.0
Mistral-24B · ridge,0.735 → 0.736,0.554 → 0.562,15.2 → 14.6,75.2 → 73.9
Mistral-24B · ridge_outlet,0.730 → 0.736,0.551 → 0.561,15.2 → 14.9,75.2 → 73.5
Mistral-24B · mlp,0.749 → 0.730,0.575 → 0.557,14.2 → 14.8,74.8 → 73.9


In [10]:
table("all", "normal")

measure,Spearman,Kendall,MAE,within 20
model,,,,
TF-IDF,0.691 → 0.706,0.496 → 0.511,17.7 → 17.5,61.1 → 60.2
Qwen3-8B · ridge,0.721 → 0.736,0.525 → 0.544,16.1 → 15.5,69.0 → 71.2
Qwen3-8B · ridge_outlet,0.724 → 0.726,0.525 → 0.534,15.7 → 15.8,70.4 → 69.5
Qwen3-8B · mlp,0.717 → 0.714,0.521 → 0.525,16.4 → 16.5,67.7 → 67.3
Mistral-24B · ridge,0.735 → 0.738,0.538 → 0.551,15.7 → 15.4,68.6 → 73.5 ✓
Mistral-24B · ridge_outlet,0.737 → 0.740,0.544 → 0.550,15.6 → 15.5,69.0 → 73.9 ✓
Mistral-24B · mlp,0.751 → 0.736,0.563 → 0.553,14.8 → 15.2,73.9 → 73.9


In [11]:
table("5", "stretched")

measure,Spearman,Kendall,MAE,within 20
model,,,,
TF-IDF,0.609 → 0.619,0.443 → 0.450,18.5 → 18.6,65.5 → 64.1
Qwen3-8B · ridge,0.663 → 0.682,0.488 → 0.507,17.1 → 16.5,69.5 → 70.4
Qwen3-8B · ridge_outlet,0.663 → 0.669,0.486 → 0.494,17.3 → 16.9,68.3 → 69.1
Qwen3-8B · mlp,0.674 → 0.672,0.498 → 0.500,16.9 → 16.8,69.7 → 69.4
Mistral-24B · ridge,0.712 → 0.718,0.533 → 0.544,16.0 → 15.5,71.4 → 72.6
Mistral-24B · ridge_outlet,0.710 → 0.710,0.531 → 0.533,16.0 → 15.9,71.7 → 70.5
Mistral-24B · mlp,0.721 → 0.711,0.546 → 0.539,15.2 → 15.5,73.3 → 71.3


In [12]:
table("1", "stretched")

measure,Spearman,Kendall,MAE,within 20
model,,,,
TF-IDF,0.420 → 0.432,0.301 → 0.309,22.4 → 22.3,57.2 → 57.4
Qwen3-8B · ridge,0.509 → 0.529,0.365 → 0.383,20.9 → 20.3,60.4 → 61.6
Qwen3-8B · ridge_outlet,0.507 → 0.513,0.365 → 0.371,20.7 → 20.5,61.2 → 60.4
Qwen3-8B · mlp,0.547 → 0.536,0.398 → 0.390,19.9 → 20.1,62.2 → 61.6
Mistral-24B · ridge,0.627 → 0.633,0.459 → 0.470,18.1 → 17.6,66.2 → 68.3 ✓
Mistral-24B · ridge_outlet,0.629 → 0.619,0.461 → 0.456,18.1 → 18.1,67.1 → 65.6
Mistral-24B · mlp,0.627 → 0.627,0.465 → 0.465,17.7 → 17.7,67.4 → 67.5


## In short

In [13]:
d = T[(T.k == "all") & (T.score == "stretched")]
print(d.pivot_table(index="model", columns="measure", values=["without", "with extra"]).round(3).to_string())
print("\nsignificantly better with extra outlets:", d.better.sum(), "| worse:", d.worse.sum(), "| of", len(d))

                           with extra                            without                           
measure                       Kendall     MAE Spearman within 20 Kendall     MAE Spearman within 20
model                                                                                              
Mistral-24B · mlp               0.557  14.764    0.730    73.894   0.575  14.197    0.749    74.779
Mistral-24B · ridge             0.562  14.626    0.736    73.894   0.554  15.248    0.735    75.221
Mistral-24B · ridge_outlet      0.561  14.917    0.736    73.451   0.551  15.177    0.730    75.221
Qwen3-8B · mlp                  0.544  15.146    0.719    73.009   0.539  15.651    0.718    76.106
Qwen3-8B · ridge                0.563  14.552    0.739    74.336   0.544  15.293    0.726    75.664
Qwen3-8B · ridge_outlet         0.543  15.351    0.723    71.681   0.543  15.629    0.726    73.009
TF-IDF                          0.529  16.507    0.710    68.584   0.518  16.287    0.696    70.796
